In [1]:
import pandas as pd
import numpy as np
import torch.nn as nn
from model import LSTMv1, LSTMv2
from torch.utils import data
import torch
from sklearn.model_selection import train_test_split
from IPython.display import clear_output 
import random
import string

# But du notebook :
Le but est de reproduire partiellement la démarche d'Andrej Karpathy, à savoir : utiliser un RNN pour générer du texte. 

Deux implémentations de RNNs (plus précisément de LSTMs) sont utilisées :
- LSTMv1 : utilisation de l'implémentation de Pytorch ```nn.LSTM()``` (c'est juste un lstm avec une couche _fully connected_)
- LSTMv2 : implémentation du LSTM "à la main". Ce modèle est moins efficace que l'implémentation dans la librairie Pytorch, mais le but ici est surtout pédagogique (je voulais voir et comprendre les équations utilisées)

Le tout est appliqué à la génération de noms de communes françaises ! J'ai récupéré la liste des communes [ici](https://www.data.gouv.fr/datasets/liste-des-communes-de-france-code-insee-codes-postaux-epci-population-superficie-62-indicateurs), les noms sont formatés pour ne comprendre que des lettres minuscules. Les accents, apostrophes et tirets sont gardés ce qui fait que 45 symboles sont nécéssaires pour écrire les noms des différentes communes. Un 46ème caractère, '#', est ajouté pour marquer la fin des mots. Il permet, pendant la phase de génération, de détecter le moment où il faut arrêter la génération de caractères.

## Lecture dataset, création d'un vocabulaire et des encodages

In [2]:
noms = pd.read_csv("communes-france-2026.csv", dtype=str)["nom_standard"]
noms = noms.to_list()
for i, nom in enumerate(noms):
    noms[i] = nom.lower() + '#' # '#' = caractère de fin de mot

In [3]:
listVocab = [] # liste du vocabulaire, ici le vocabulaire = les lettres possibles
for nom in noms:
    for letter in nom:
        if letter not in listVocab:
            listVocab.append(letter)
i = 0
listVocab.sort()
while listVocab[i] != 'a':
    i += 1
listVocab[:-i], listVocab[-i:] = listVocab[i:], listVocab[0:i] # juste pour avoir la liste qui commence par a et qui finit par les caractères spéciaux

In [4]:
nVocab = len(listVocab)
nVocab

46

In [5]:
encodings = np.eye(nVocab) # encoding i = que des 0 sauf l'élt i qui vaut 1
dicoEncodings = {}
inverseEncodings = {}
for i, letter in enumerate(listVocab):
    dicoEncodings[letter] = encodings[i]
    inverseEncodings[i] = letter

# Entrainement du LTSM

In [281]:
rnn = LSTMv1(2, 80, nVocab, dropout=0.2)
rnn = LSTMv2(2, 30, nVocab)

In [68]:
def encode(listOfWords):
    encoded = []
    for word in listOfWords:
        encoding = []
        for letter in word:
            encoding.append(dicoEncodings[letter])
        encoded.append(np.array(encoding))
    return encoded

In [69]:
nomsEncoded = encode(noms)

In [70]:
X_train, X_test = train_test_split(nomsEncoded, test_size=0.2, shuffle=True)

In [282]:
optimizer = torch.optim.Adam(rnn.parameters(), lr=5e-3)
criterion = nn.CrossEntropyLoss()
nEpoch = 5
batchSize = 64
loss_train_list = []
loss_valid_list = []
for epoch in range(nEpoch):
    random.shuffle(X_train)
    total_loss = 0
    print(epoch)
    loss_train = 0
    loss_valid = 0
    n_Batch = 0
    rnn.train()
    for i, x in enumerate(X_train):
        xTensor = torch.tensor(x).float()
        if xTensor.shape[0] < 2: #il y a quand même une ville qui s'appelle "y"
            continue
        target = xTensor[1:]
        target = target.argmax(dim=1)
        inputRnn = xTensor[:-1]
        pred = rnn(inputRnn)
        loss = criterion(pred, target)
        loss_train += loss.item()
        loss.backward()
        n_Batch += 1
        if n_Batch == batchSize:
            optimizer.step()
            optimizer.zero_grad()
            n_Batch = 0
        if i%1000 == 0:
            clear_output(wait=True)
            print("List loss entrainement :", np.array(loss_train_list))
            print("List loss validation :", np.array(loss_valid_list))
            print("Epoch : ", epoch, " Itération : ", i)
    if n_Batch != 0:
        optimizer.step()
        optimizer.zero_grad()
    with torch.no_grad():
        rnn.eval()
        for i, x in enumerate(X_test):
            xTensor = torch.tensor(x).float()
            if xTensor.shape[0] < 2: #il y a quand même une ville qui s'appelle "y"
                continue
            target = xTensor[1:]
            target = target.argmax(dim=1)
            inputRnn = xTensor[:-1]
            pred = rnn(inputRnn)
            loss = criterion(pred, target)
            loss_valid += loss.item()
    loss_train_list.append(np.round(loss_train, decimals=0))
    loss_valid_list.append(np.round(loss_valid, decimals=0))

    

List loss entrainement : [72835. 58585. 55321. 53739.]
List loss validation : [15387. 14190. 13718. 13421.]
Epoch :  4  Itération :  27000


# Essais de génération de noms de communes

In [12]:
def generateName(firstLetter=None, temperature = 1):
    if firstLetter == None:
        firstLetter = (random.choice(string.ascii_letters)).lower()
    rnn.eval()
    startEncoded = [dicoEncodings[firstLetter]]
    result = firstLetter
    nextChar = ""
    temperature = temperature
    while nextChar != '#':
        nextChar = rnn(torch.tensor(startEncoded).float())[-1]
        probs = torch.softmax(nextChar/temperature, dim=-1)
        nextChar = torch.multinomial(probs, num_samples=1)
        nextChar = inverseEncodings[nextChar.item()]
        startEncoded.append(dicoEncodings[nextChar])
        result += nextChar
    return result[:-1]

In [319]:
nInTrainingData = 0
for i in range(1000):
    name = generateName(temperature=1) + '#'
    if name in noms:
        nInTrainingData += 1
print("Sur 1000 données générées aléatoirement,", nInTrainingData, "sont incluses dans le dataset d'entraînement")

Sur 1000 données générées aléatoirement, 29 sont incluses dans le dataset d'entraînement


In [356]:
result = generateName(temperature=1)
result

'pourville'

In [357]:
(result + '#') in noms

False

# Quelques résultats

De façon qualitative, les noms générés correspondent assez bien à ce à quoi je m'attendais. J'ai vérifié que le RRN ne se contente pas de donner des noms appris par coeur : en général (dans environ 95% des cas), les noms produits ne sont pas dans le dataset de départ.

Voici enfin une liste de quelques noms de communes qui malheureusement n'existent pas en France :

| Nom | Température |
| --- | --- |
| Chalotte-sur-sevicourt | 1 |
| Offreville| 1 |
| Petit-sur-brenonne | 1 |
| Vinournolle | 1 |
| Espiras | 0.5 |
| Saint-martin-de-colombe | 0.5 |
| Sauvillon | 1.5 |
| Gruvaux-bonnay-aux-bicles | 1.2 |
| Ziveville-en-thieux | 1.2 |

Augmenter la température au-delà de 1.5 donne des résultats un peu trop originaux, comme "Xeix-ommelonson", "Rojay-teiltultesulcébeville" ou encore "Kaèuguerièraeazrell" (ça devient difficile à prononcer)

Il serait probablement facile de faire plus de tests/mesures/statistiques pour évaluer les performances du modèle. De même, le code est probablement optimisable et améliorable, mais j'ai fait ce notebook surtout dans le but de m'amuser, et par soucis de temps, je compte en rester là pour l'instant !